In [1]:
# Import libraries
import json
from pathlib import Path
from scipy import stats
import numpy as np
import pandas as pd

print("✓ Libraries loaded")

✓ Libraries loaded


In [3]:
# Load evaluation results
BASE_DIR = Path.cwd().parent
RESULTS = {
    "yolo": BASE_DIR / "outputs/predictions/07_e2e_yolov8_paper_faithful_20260820-140039/report_test_conf80_beta.json",
    "baseline": BASE_DIR / "outputs/predictions/06a_faster_rcnn_eval_20260815-235616/report_test_conf80_beta.json",
    "sam": BASE_DIR / "outputs/predictions/06b_faster_rcnn_sam_eval_20260816-102627/report_test_conf80_beta.json"
}

def load_per_class_mae(json_path):
    """Extract per-class MAE from JSON report"""
    with open(json_path) as f:
        data = json.load(f)
    return {
        "class_names": [cls["class_name"] for cls in data["per_class"]],
        "mae": [cls["abs_me_volume_pct"] for cls in data["per_class"]],
        "overall_mae": data["overall"]["mean_abs_me_volume_pct"]
    }

# Load all results
yolo_data = load_per_class_mae(RESULTS["yolo"])
baseline_data = load_per_class_mae(RESULTS["baseline"])
sam_data = load_per_class_mae(RESULTS["sam"])

print(f"✓ Loaded {len(yolo_data['mae'])} classes for each pipeline")
print(f"  YOLO overall MAE:     {yolo_data['overall_mae']:.2f}%")
print(f"  Baseline overall MAE: {baseline_data['overall_mae']:.2f}%")
print(f"  SAM overall MAE:      {sam_data['overall_mae']:.2f}%")

✓ Loaded 19 classes for each pipeline
  YOLO overall MAE:     18.26%
  Baseline overall MAE: 19.97%
  SAM overall MAE:      17.99%


In [4]:
# Create per-class comparison table
df_comparison = pd.DataFrame({
    "Class": yolo_data["class_names"],
    "YOLO": yolo_data["mae"],
    "Baseline": baseline_data["mae"],
    "SAM": sam_data["mae"]
})

df_comparison["YOLO vs Baseline"] = df_comparison["YOLO"] - df_comparison["Baseline"]
df_comparison["Winner"] = df_comparison.apply(
    lambda row: "YOLO" if row["YOLO"] < row["Baseline"] else "Baseline" if row["YOLO"] > row["Baseline"] else "Tie",
    axis=1
)

# Display
print("Per-class MAE comparison (%):\n")
print(df_comparison.to_string(index=False))
print(f"\nWin/Loss record:")
print(df_comparison["Winner"].value_counts())

Per-class MAE comparison (%):

            Class      YOLO  Baseline       SAM  YOLO vs Baseline   Winner
            apple 13.432574 19.461937 18.098126         -6.029363     YOLO
           banana 22.696860 22.649545 18.659503          0.047316 Baseline
            bread 32.022638 30.774200 27.476144          1.248438 Baseline
              bun 17.544760 24.597873 18.849893         -7.053113     YOLO
         doughnut 14.594844 18.766493 17.301637         -4.171650     YOLO
              egg 32.509468 26.205598 27.303255          6.303869 Baseline
fried_dough_twist 25.743387 23.374197 22.026754          2.369190 Baseline
            grape 26.714809 43.060242 19.343977        -16.345433     YOLO
            lemon 16.945349 17.353689 17.300290         -0.408340     YOLO
           litchi  9.810481  9.842534  9.680967         -0.032053     YOLO
            mango 20.776422 23.005163 22.596398         -2.228741     YOLO
         mooncake 14.033974 13.516097 12.631304          0.517877 Bas

In [5]:
# Statistical Test: YOLO vs Baseline
print("=" * 70)
print("Statistical Test: YOLO26-seg vs Baseline (Faster R-CNN + GrabCut)")
print("=" * 70)

yolo_mae = yolo_data["mae"]
baseline_mae = baseline_data["mae"]

# Paired t-test
t_stat, p_ttest = stats.ttest_rel(yolo_mae, baseline_mae)

# Wilcoxon signed-rank test (non-parametric)
w_stat, p_wilcoxon = stats.wilcoxon(yolo_mae, baseline_mae)

# Effect size and confidence interval
diff = np.array(yolo_mae) - np.array(baseline_mae)
mean_diff = np.mean(diff)
std_diff = np.std(diff, ddof=1)
ci_95 = 1.96 * std_diff / np.sqrt(len(diff))

yolo_mean = np.mean(yolo_mae)
baseline_mean = np.mean(baseline_mae)

print(f"\n📊 Descriptive Statistics:")
print(f"  YOLO26-seg MAE:  {yolo_mean:.2f}% (SD = {np.std(yolo_mae, ddof=1):.2f})")
print(f"  Baseline MAE:    {baseline_mean:.2f}% (SD = {np.std(baseline_mae, ddof=1):.2f})")
print(f"  Mean difference: {mean_diff:.2f}pp")
print(f"  95% CI:          [{mean_diff - ci_95:.2f}, {mean_diff + ci_95:.2f}]")

print(f"\n🧪 Statistical Tests:")
print(f"  Paired t-test:   t({len(yolo_mae)-1}) = {t_stat:.3f}, p = {p_ttest:.4f}")
print(f"  Wilcoxon test:   W = {w_stat:.1f}, p = {p_wilcoxon:.4f}")

print(f"\n✅ Verdict (α = 0.05):")
if p_ttest < 0.05:
    print(f"  ✓ Statistically SIGNIFICANT (p = {p_ttest:.4f})")
else:
    print(f"  ✗ NOT statistically significant (p = {p_ttest:.4f})")

print(f"\nInterpretation: YOLO26-seg MAE is {'lower' if mean_diff < 0 else 'higher'} than baseline by {abs(mean_diff):.2f}pp.")

Statistical Test: YOLO26-seg vs Baseline (Faster R-CNN + GrabCut)

📊 Descriptive Statistics:
  YOLO26-seg MAE:  18.26% (SD = 7.20)
  Baseline MAE:    19.97% (SD = 7.90)
  Mean difference: -1.70pp
  95% CI:          [-3.85, 0.44]

🧪 Statistical Tests:
  Paired t-test:   t(18) = -1.558, p = 0.1365
  Wilcoxon test:   W = 65.0, p = 0.2413

✅ Verdict (α = 0.05):
  ✗ NOT statistically significant (p = 0.1365)

Interpretation: YOLO26-seg MAE is lower than baseline by 1.70pp.


In [6]:
# Statistical Test: YOLO vs SAM
print("=" * 70)
print("Statistical Test: YOLO26-seg vs Faster R-CNN + SAM")
print("=" * 70)

sam_mae = sam_data["mae"]

# Paired t-test
t_stat2, p_ttest2 = stats.ttest_rel(yolo_mae, sam_mae)

# Wilcoxon test
w_stat2, p_wilcoxon2 = stats.wilcoxon(yolo_mae, sam_mae)

# Effect size and CI
diff2 = np.array(yolo_mae) - np.array(sam_mae)
mean_diff2 = np.mean(diff2)
std_diff2 = np.std(diff2, ddof=1)
ci_95_2 = 1.96 * std_diff2 / np.sqrt(len(diff2))

sam_mean = np.mean(sam_mae)

print(f"\n📊 Descriptive Statistics:")
print(f"  YOLO26-seg MAE:  {yolo_mean:.2f}%")
print(f"  SAM MAE:         {sam_mean:.2f}%")
print(f"  Mean difference: {mean_diff2:.2f}pp")
print(f"  95% CI:          [{mean_diff2 - ci_95_2:.2f}, {mean_diff2 + ci_95_2:.2f}]")

print(f"\n🧪 Statistical Tests:")
print(f"  Paired t-test:   t({len(yolo_mae)-1}) = {t_stat2:.3f}, p = {p_ttest2:.4f}")
print(f"  Wilcoxon test:   W = {w_stat2:.1f}, p = {p_wilcoxon2:.4f}")

print(f"\n✅ Verdict (α = 0.05):")
if p_ttest2 < 0.05:
    print(f"  ✓ Statistically SIGNIFICANT (p = {p_ttest2:.4f})")
else:
    print(f"  ✗ NOT statistically significant (p = {p_ttest2:.4f})")

print(f"\nInterpretation: YOLO26-seg MAE is {'lower' if mean_diff2 < 0 else 'higher'} than SAM by {abs(mean_diff2):.2f}pp.")
print("\n✅ Analysis complete!")

Statistical Test: YOLO26-seg vs Faster R-CNN + SAM

📊 Descriptive Statistics:
  YOLO26-seg MAE:  18.26%
  SAM MAE:         17.99%
  Mean difference: 0.27pp
  95% CI:          [-1.25, 1.80]

🧪 Statistical Tests:
  Paired t-test:   t(18) = 0.353, p = 0.7282
  Wilcoxon test:   W = 90.0, p = 0.8596

✅ Verdict (α = 0.05):
  ✗ NOT statistically significant (p = 0.7282)

Interpretation: YOLO26-seg MAE is higher than SAM by 0.27pp.

✅ Analysis complete!
